# 1. Importing the Dataset

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rtatman/ubuntu-dialogue-corpus")

print("Path to dataset files:", path)

/home/arhan.khade/.conda/envs/prosper/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /home/arhan.khade/.cache/kagglehub/datasets/rtatman/ubuntu-dialogue-corpus/versions/2
Path to dataset files: /home/arhan.khade/.cache/kagglehub/datasets/rtatman/ubuntu-dialogue-corpus/versions/2


In [2]:
import pandas as pd

path = "/home/arhan.khade/.cache/kagglehub/datasets/rtatman/ubuntu-dialogue-corpus/versions/2/Ubuntu-dialogue-corpus/dialogueText.csv"

df = pd.read_csv(path)
print(df.head())

   folder  dialogueID                      date       from         to  \
0       3  126125.tsv  2008-04-23T14:55:00.000Z  bad_image        NaN   
1       3  126125.tsv  2008-04-23T14:56:00.000Z  bad_image        NaN   
2       3  126125.tsv  2008-04-23T14:57:00.000Z  lordleemo  bad_image   
3       3   64545.tsv  2009-08-01T06:22:00.000Z   mechtech        NaN   
4       3   64545.tsv  2009-08-01T06:22:00.000Z   mechtech        NaN   

                                                text  
0  Hello folks, please help me a bit with the fol...  
1  Did I choose a bad channel? I ask because you ...  
2  the second sentence is better english   and we...  
3                                       Sock Puppe?t  
4                                               WTF?  


In [3]:
print(df.columns)
print(df.shape)

Index(['folder', 'dialogueID', 'date', 'from', 'to', 'text'], dtype='str')
(1038324, 6)


# 2. Cutting down to 5k dialogueIDs

In [4]:
import pandas as pd

# Load the data
df = pd.read_csv(path)

# Drop any rows where text is missing
df = df.dropna(subset=['text'])

# 1. Grab a random sample of 5,000 unique dialogueIDs
sample_ids = df['dialogueID'].drop_duplicates().sample(n=5000, random_state=42)

# 2. Filter your dataframe to only include those dialogues
df_dev = df[df['dialogueID'].isin(sample_ids)].copy()

In [5]:
# Create a new column that formats the chat log
# Example output: "user123: how do i install python?"
df_dev['formatted_text'] = df_dev['from'].astype(str) + ": " + df_dev['text'].astype(str)

In [6]:
# Ensure the conversation is in chronological order
df_dev = df_dev.sort_values(by=['dialogueID', 'date'])

# Group by the dialogue ID and stitch the formatted text together
conversations = df_dev.groupby('dialogueID').agg(
    full_text=('formatted_text', lambda x: '\n'.join(x)),
    folder=('folder', 'first'),       # Keep the category as metadata
    start_date=('date', 'min'),       # Track when it started
    end_date=('date', 'max')          # Track when it ended
).reset_index()

# 3. Chunking

In [7]:
from langchain_text_splitters  import RecursiveCharacterTextSplitter

# Create a text splitter
# chunk_size is roughly characters (not tokens, but a good proxy)
# chunk_overlap ensures context isn't lost if a sentence is split in half
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ".", " "]
)

final_chunks = []
final_metadata = []

# Iterate through your grouped conversations
for index, row in conversations.iterrows():
    # Split the long conversation into smaller pieces
    chunks = text_splitter.split_text(row['full_text'])
    
    for i, chunk in enumerate(chunks):
        final_chunks.append(chunk)
        # Store metadata so we know exactly where this chunk came from
        final_metadata.append({
            "dialogueID": row['dialogueID'],
            "folder": row['folder'],
            "chunk_index": i
        })

In [8]:
chunks

['rsc-: sudo apt-get install ntfsprogs\nrsc-: wow, do not even try to do a chkdsk over wine unless you want disaster to happen :P\nk0de: lol no joke']